# YOLO26 object detection

Runs Ultralytics YOLO26 on an image, a video file or a folder of frames. Weights are downloaded on first use.

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
from pathlib import Path

import cv2
import torch
from IPython.display import Image as IPImage, display
from ultralytics import YOLO

## Model

Sizes: `yolo26n` (fastest) < `s` < `m` < `l` < `x` (most accurate). Use a `-seg`, `-pose`, `-obb` or `-cls` suffix for the other tasks.

In [ ]:
MODEL_NAME = "yolo26n.pt"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

model = YOLO(MODEL_NAME)  # downloads the weights if they are not already here
print(f"device: {DEVICE}, classes: {len(model.names)}")

## Detect on an image

`SOURCE` can be a local path or a URL.

In [ ]:
SOURCE = "https://ultralytics.com/images/bus.jpg"
CONF = 0.25

results = model.predict(SOURCE, conf=CONF, device=DEVICE, verbose=False)
r = results[0]

for box in r.boxes:
    x1, y1, x2, y2 = (round(v) for v in box.xyxy[0].tolist())
    print(f"{model.names[int(box.cls)]:<12} {float(box.conf):.2f}  ({x1}, {y1}, {x2}, {y2})")

display(IPImage(data=cv2.imencode(".jpg", r.plot())[1].tobytes()))

## Detect on a video or a folder of images

Writes the annotated result to `runs/` and shows the per-class counts for each frame. Point `VIDEO_SOURCE` at a video file, a folder of images, or `0` for a webcam.

In [ ]:
VIDEO_SOURCE = "./data/sample.mp4"

if not (VIDEO_SOURCE == 0 or Path(VIDEO_SOURCE).exists()):
    print(f"{VIDEO_SOURCE} not found: put a video or image folder there, or change VIDEO_SOURCE")
else:
    for i, r in enumerate(model.predict(VIDEO_SOURCE, conf=CONF, device=DEVICE, stream=True,
                                        save=True, project="runs", name="yolo26", exist_ok=True,
                                        verbose=False)):
        counts = {}
        for c in r.boxes.cls.tolist():
            counts[model.names[int(c)]] = counts.get(model.names[int(c)], 0) + 1
        print(f"frame {i}: {counts or 'nothing detected'}")
    print(f"saved to {Path('runs/yolo26').resolve()}")